# LeetCode #106 — Construct Binary Tree from Inorder and Postorder Traversal: gốc nằm ở cuối

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, Divide and Conquer, Tree, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def buildTree(self, inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
```

Bài này là "anh em" của #105. Khác biệt duy nhất: gốc lấy từ **cuối** postorder thay vì đầu preorder. Nhưng chính chi tiết đó buộc ta **đổi thứ tự dựng** — và đó là chỗ dễ sai nhất.

Mục tiêu của notebook:

1. Hiểu postorder: trái → phải → **gốc**, nên gốc của mỗi cây con nằm ở **cuối** đoạn của nó.
2. Đọc postorder **từ phải sang trái** = "gốc → phải → trái" — nên phải dựng cây **phải trước**.
3. Dùng hash map vị trí inorder để đạt `O(n)`.
4. Cách thay thế: truyền khoảng của **cả hai** mảng (không cần con trỏ toàn cục).

# 1. Bài toán là gì?

Cho `inorder` và `postorder` của **cùng một** cây nhị phân, dựng lại cây.

- Inorder: trái → **gốc** → phải.
- Postorder: trái → phải → **gốc**.

**Constraints (theo đề):**

- `1 <= inorder.length <= 3000`, `postorder.length == inorder.length`
- `-3000 <= inorder[i], postorder[i] <= 3000`
- Mọi giá trị **phân biệt**.
- Mọi giá trị của `postorder` đều có trong `inorder`; đảm bảo hai mảng là inorder/postorder của cùng một cây.

---

## Ví dụ

```text
inorder   = [9, 3, 15, 20, 7]
postorder = [9, 15, 7, 20, 3]
->            3
             / \
            9  20         [3, 9, 20, null, null, 15, 7]
              /  \
             15   7

inorder = [-1], postorder = [-1]  ->  [-1]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Đệ quy (recursion)
3. Chia để trị (divide and conquer)
4. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.3 Chia để trị (divide and conquer)

### Ý tưởng

1. **Chia** bài toán thành các bài con **cùng dạng**, nhỏ hơn (thường là hai nửa).
2. **Trị**: giải các bài con bằng đệ quy (bài đủ nhỏ thì giải trực tiếp).
3. **Kết hợp** kết quả các bài con thành kết quả bài lớn.

Khác DP: các bài con **không chồng lấp** (mỗi phần dữ liệu thuộc đúng một bài con).

```text
            [8 phần tử]                      tầng 0: 1 bài, tổng n việc
         [4]          [4]                    tầng 1: 2 bài, tổng n việc
       [2]  [2]     [2]  [2]                 tầng 2: 4 bài, tổng n việc
      [1][1][1][1] [1][1][1][1]              log₂ n tầng  →  O(n log n)
```

In [11]:
def max_chia_de_tri(a, lo, hi, sau=0):
    """Phần tử lớn nhất của a[lo..hi]."""
    if lo == hi:
        return a[lo]                                   # bài đủ nhỏ
    mid = (lo + hi) // 2
    trai = max_chia_de_tri(a, lo, mid, sau + 1)        # trị nửa trái
    phai = max_chia_de_tri(a, mid + 1, hi, sau + 1)    # trị nửa phải
    kq = max(trai, phai)                               # kết hợp
    print("    " * sau + f"a[{lo}..{hi}] = {a[lo:hi + 1]} → max {kq}")
    return kq

_ = max_chia_de_tri([3, 8, 1, 9, 4, 7, 2, 6], 0, 7)

        a[0..1] = [3, 8] → max 8
        a[2..3] = [1, 9] → max 9
    a[0..3] = [3, 8, 1, 9] → max 9
        a[4..5] = [4, 7] → max 7
        a[6..7] = [2, 6] → max 6
    a[4..7] = [4, 7, 2, 6] → max 7
a[0..7] = [3, 8, 1, 9, 4, 7, 2, 6] → max 9


### Các ví dụ quen thuộc

| Bài | Chia | Kết hợp | Độ phức tạp |
|---|---|---|---|
| Tìm nhị phân | bỏ một nửa | không cần | `O(log n)` |
| Merge sort | hai nửa | trộn `O(n)` | `O(n log n)` |
| Lũy thừa nhanh `x^n` | `x^(n/2)` | bình phương | `O(log n)` |
| Dựng cây cân bằng từ mảng sắp xếp | phần tử giữa làm gốc | gắn hai cây con | `O(n)` |
| Dựng cây từ hai cách duyệt | tách theo vị trí gốc | gắn hai cây con | `O(n)` với hash map |

In [12]:
def luy_thua_nhanh(x, n, sau=0):
    if n == 0:
        return 1
    nua = luy_thua_nhanh(x, n // 2, sau + 1)
    kq = nua * nua * (x if n % 2 else 1)
    print("    " * sau + f"{x}^{n} = ({x}^{n // 2})²{' · ' + str(x) if n % 2 else ''} = {kq}")
    return kq

_ = luy_thua_nhanh(2, 10)

            2^1 = (2^0)² · 2 = 2
        2^2 = (2^1)² = 4
    2^5 = (2^2)² · 2 = 32
2^10 = (2^5)² = 1024


Lỗi hay gặp: chia không đều hoặc không nhỏ đi (`mid` tính sai → đệ quy vô hạn); tạo bản sao `a[:mid]` ở mỗi tầng (tốn thêm `O(n log n)` bộ nhớ — truyền chỉ số `lo, hi` thay vì cắt list); quên trường hợp đoạn rỗng.

## 2.4 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [13]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [14]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 376.35 ms | trên set: 0.082 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [15]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [16]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.5 Áp dụng vào bài này

Giống #105 nhưng dùng **postorder** (gốc đứng **cuối**) và **inorder**. Lấy gốc từ cuối postorder, tra vị trí trong inorder bằng **hash map** để tách hai cây con, rồi **đệ quy** — nếu lấy từ cuối postorder thì phải dựng cây con **phải trước**.

# 3. Nền tảng

In [17]:
from typing import List, Optional
from collections import deque
import random, sys
sys.setrecursionlimit(10000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def serialize(root: Optional[TreeNode]) -> list:
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out


def inorder_of(root):
    return inorder_of(root.left) + [root.val] + inorder_of(root.right) if root else []


def postorder_of(root):
    return postorder_of(root.left) + postorder_of(root.right) + [root.val] if root else []

t = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))
print("inorder:", inorder_of(t), "| postorder:", postorder_of(t))

inorder: [9, 3, 15, 20, 7] | postorder: [9, 15, 7, 20, 3]


# 4. Ý tưởng chính

```text
inorder   = [ 9 | 3 | 15 20 7 ]          postorder = [ 9 | 15 7 20 | 3 ]
             trái gốc   phải                          trái   phải   gốc
```

1. `postorder[-1]` là **gốc**.
2. Vị trí `k` của gốc trong `inorder` chia trái / phải. `left_size = k − in_lo`.
3. Trong postorder: `left_size` phần tử **đầu** là cây trái, phần giữa (trước gốc) là cây phải.
4. Đệ quy.

Đọc postorder **ngược** từ cuối: `3, 20, 7, 15, 9` — đó là "gốc → **phải** → trái". Nếu dùng một con trỏ chạy ngược trên postorder, ta phải dựng **cây phải trước** để con trỏ tiêu thụ đúng thứ tự.

In [18]:
post = [9, 15, 7, 20, 3]
print("postorder đọc ngược:", post[::-1])

def root_right_left(root):
    return [root.val] + root_right_left(root.right) + root_right_left(root.left) if root else []

print("duyệt gốc → phải → trái:", root_right_left(t))

postorder đọc ngược: [3, 20, 7, 15, 9]
duyệt gốc → phải → trái: [3, 20, 7, 15, 9]


# 5. Baseline: cắt list

In [19]:
def build_slicing(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    if not postorder:
        return None
    root_val = postorder[-1]
    k = inorder.index(root_val)
    root = TreeNode(root_val)
    root.left = build_slicing(inorder[:k], postorder[:k])
    root.right = build_slicing(inorder[k + 1:], postorder[k:-1])
    return root

print(serialize(build_slicing([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])))

[3, 9, 20, None, None, 15, 7]


`O(n²)` xấu nhất vì cắt list và `index` ở mỗi tầng — nhưng rất dễ đọc, và cho phép dựng trái hay phải trước tùy ý (vì mỗi lời gọi nhận đúng đoạn của nó).

# 6. Lời giải chính: hash map + con trỏ chạy ngược

In [20]:
def build_hash(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1

    def build(in_lo, in_hi):
        nonlocal post_idx
        if in_lo > in_hi:
            return None
        root_val = postorder[post_idx]
        post_idx -= 1
        k = pos[root_val]
        root = TreeNode(root_val)
        root.right = build(k + 1, in_hi)       # PHẢI trước
        root.left = build(in_lo, k - 1)
        return root

    return build(0, len(inorder) - 1)

print(serialize(build_hash([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])))

[3, 9, 20, None, None, 15, 7]


# 7. Mô phỏng

In [21]:
def trace_build(inorder: List[int], postorder: List[int]) -> list:
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1
    def build(in_lo, in_hi, depth):
        nonlocal post_idx
        if in_lo > in_hi:
            return None
        root_val = postorder[post_idx]
        k = pos[root_val]
        print("  " * depth + f"postorder[{post_idx}] = {root_val} là gốc của inorder{inorder[in_lo:in_hi + 1]}"
              f" -> phải {inorder[k + 1:in_hi + 1]}, trái {inorder[in_lo:k]}")
        post_idx -= 1
        root = TreeNode(root_val)
        root.right = build(k + 1, in_hi, depth + 1)
        root.left = build(in_lo, k - 1, depth + 1)
        return root
    root = build(0, len(inorder) - 1, 0)
    print("Cây:", serialize(root))
    return serialize(root)

_ = trace_build([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
print()
_ = trace_build([4, 2, 5, 1, 6, 3, 7], [4, 5, 2, 6, 7, 3, 1])

postorder[4] = 3 là gốc của inorder[9, 3, 15, 20, 7] -> phải [15, 20, 7], trái [9]
  postorder[3] = 20 là gốc của inorder[15, 20, 7] -> phải [7], trái [15]
    postorder[2] = 7 là gốc của inorder[7] -> phải [], trái []
    postorder[1] = 15 là gốc của inorder[15] -> phải [], trái []
  postorder[0] = 9 là gốc của inorder[9] -> phải [], trái []
Cây: [3, 9, 20, None, None, 15, 7]

postorder[6] = 1 là gốc của inorder[4, 2, 5, 1, 6, 3, 7] -> phải [6, 3, 7], trái [4, 2, 5]
  postorder[5] = 3 là gốc của inorder[6, 3, 7] -> phải [7], trái [6]
    postorder[4] = 7 là gốc của inorder[7] -> phải [], trái []
    postorder[3] = 6 là gốc của inorder[6] -> phải [], trái []
  postorder[2] = 2 là gốc của inorder[4, 2, 5] -> phải [5], trái [4]
    postorder[1] = 5 là gốc của inorder[5] -> phải [], trái []
    postorder[0] = 4 là gốc của inorder[4] -> phải [], trái []
Cây: [1, 2, 3, 4, 5, 6, 7]


# 8. Cách 3: truyền khoảng của cả hai mảng

Không dùng con trỏ toàn cục: mỗi lời gọi nhận `(in_lo, in_hi, post_lo, post_hi)`. Gốc là `postorder[post_hi]`. Cây trái chiếm `postorder[post_lo : post_lo + left_size]`, cây phải chiếm phần còn lại trước gốc. Thứ tự dựng trái/phải **không** còn quan trọng.

In [22]:
def build_ranges(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    def build(in_lo, in_hi, post_lo, post_hi):
        if in_lo > in_hi:
            return None
        root = TreeNode(postorder[post_hi])
        k = pos[root.val]
        left_size = k - in_lo
        root.left = build(in_lo, k - 1, post_lo, post_lo + left_size - 1)
        root.right = build(k + 1, in_hi, post_lo + left_size, post_hi - 1)
        return root
    return build(0, len(inorder) - 1, 0, len(postorder) - 1)

print(serialize(build_ranges([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])))

[3, 9, 20, None, None, 15, 7]


# 9. Phiên bản đúng format LeetCode

In [23]:
class Solution:
    def buildTree(self, inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        pos = {v: i for i, v in enumerate(inorder)}
        post_idx = len(postorder) - 1

        def build(in_lo: int, in_hi: int) -> Optional[TreeNode]:
            nonlocal post_idx
            if in_lo > in_hi:
                return None
            root = TreeNode(postorder[post_idx])
            post_idx -= 1
            k = pos[root.val]
            root.right = build(k + 1, in_hi)
            root.left = build(in_lo, k - 1)
            return root

        return build(0, len(inorder) - 1)

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def random_tree(n):
    vals = iter(random.sample(range(-3000, 3001), n))
    def make(k):
        if k == 0:
            return None
        left = random.randint(0, k - 1)
        node = TreeNode(next(vals))
        node.left = make(left)
        node.right = make(k - 1 - left)
        return node
    return make(n)


sol = Solution()
check("Ví dụ 1", serialize(sol.buildTree([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])), [3, 9, 20, None, None, 15, 7])
check("Ví dụ 2", serialize(sol.buildTree([-1], [-1])), [-1])
check("Lệch trái", serialize(sol.buildTree([1, 2, 3], [1, 2, 3])), [3, 2, None, 1])
check("Lệch phải", serialize(sol.buildTree([1, 2, 3], [3, 2, 1])), [1, None, 2, None, 3])

random.seed(106)
for _ in range(3000):
    t = random_tree(random.randint(1, 15))
    ino, post = inorder_of(t), postorder_of(t)
    expected = serialize(t)
    for f in (sol.buildTree, build_slicing, build_ranges):
        assert serialize(f(ino, post)) == expected
print("3000 cây ngẫu nhiên: 3 cách dựng lại đúng cây gốc")

chain_in = list(range(3000)); chain_post = list(range(3000))       # cây lệch trái 3000 nút
r = sol.buildTree(chain_in, chain_post)
check("Cây lệch 3000 nút", inorder_of(r) == chain_in and postorder_of(r) == chain_post, True)

Ví dụ 1            PASS  [3, 9, 20, None, None, 15, 7]
Ví dụ 2            PASS  [-1]
Lệch trái          PASS  [3, 2, None, 1]
Lệch phải          PASS  [1, None, 2, None, 3]


3000 cây ngẫu nhiên: 3 cách dựng lại đúng cây gốc
Cây lệch 3000 nút  PASS  True


# 10. Vì sao đúng? Độ phức tạp?

Postorder của cây con `T` là `[postorder(trái) | postorder(phải) | gốc]`. Đọc ngược: `gốc, postorder(phải) đọc ngược, postorder(trái) đọc ngược`. Bằng quy nạp: nếu `post_idx` đang trỏ tới gốc của `T`, thì sau khi lấy gốc, nó trỏ tới gốc của cây **phải**; sau khi dựng xong cây phải, nó trỏ tới gốc cây **trái**.

```text
Time:  O(n)   — mỗi nút một lần, tra pos O(1)
Space: O(n)   — hash map; đệ quy O(h)
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Dựng trái trước (chép nguyên code #105)

In [25]:
def wrong_left_first(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1
    def build(in_lo, in_hi):
        nonlocal post_idx
        if in_lo > in_hi:
            return None
        root = TreeNode(postorder[post_idx]); post_idx -= 1
        k = pos[root.val]
        root.left = build(in_lo, k - 1)          # SAI: con trỏ ngược đang ở cây phải
        root.right = build(k + 1, in_hi)
        return root
    return build(0, len(inorder) - 1)

try:
    r = wrong_left_first([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])
    print("Sai :", serialize(r), "| inorder của nó:", inorder_of(r))
except Exception as e:
    print("Sai : lỗi", type(e).__name__, e)
print("Đúng:", serialize(sol.buildTree([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])))

Sai : lỗi IndexError list index out of range
Đúng: [3, 9, 20, None, None, 15, 7]


Sau khi lấy gốc `3`, con trỏ trỏ vào `20` (gốc cây **phải**), nhưng bản sai dùng nó cho cây **trái** `[9]` — đoạn inorder `[9]` không chứa `20`, nên `k = pos[20]` nằm ngoài khoảng và cây bị dựng sai hoàn toàn (hoặc lỗi).

## Lỗi 2: Cắt postorder sai biên

Cây phải là `postorder[k : -1]` (bỏ gốc ở cuối). Viết `postorder[k:]` sẽ đưa gốc vào cây phải.

In [26]:
def wrong_slice(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    if not postorder:
        return None
    k = inorder.index(postorder[-1])
    root = TreeNode(postorder[-1])
    root.left = wrong_slice(inorder[:k], postorder[:k])
    root.right = wrong_slice(inorder[k + 1:], postorder[k:])       # SAI: còn dính gốc
    return root

try:
    print("Sai :", serialize(wrong_slice([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])))
except Exception as e:
    print("Sai : lỗi", type(e).__name__, "—", e)

Sai : lỗi ValueError — 9 is not in list


## Lỗi 3: Lấy gốc ở `postorder[0]`

Phần tử đầu của postorder là nút **trái nhất ở tầng sâu** (lá đầu tiên được thăm), không phải gốc.

## Lỗi 4: Gọi `inorder.index` mỗi lần

Đúng nhưng `O(n²)` với cây lệch — dựng `pos` một lần.

# 12. Mẹo nhận diện pattern

| Cặp traversal | Gốc ở | Con trỏ | Thứ tự dựng |
|---|---|---|---|
| Preorder + Inorder (#105) | đầu | chạy xuôi | gốc → trái → phải |
| Inorder + Postorder (#106) | cuối | chạy **ngược** | gốc → **phải** → trái |
| Preorder + Postorder (#889) | đầu preorder | — | dùng `pre[1]` làm gốc cây trái để chia |

Quy tắc nhớ: **thứ tự dựng phải khớp thứ tự con trỏ đọc mảng**.

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Dựng cây từ `inorder = [2, 1, 3]`, `postorder = [2, 3, 1]` và từ `inorder = [1, 2, 3, 4]`, `postorder = [1, 3, 4, 2]`.

## Bài 2 — Điền chỗ trống

In [27]:
def exercise_2_build_tree(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    pos = {v: i for i, v in enumerate(inorder)}
    post_idx = len(postorder) - 1
    def build(in_lo, in_hi):
        nonlocal post_idx
        # TODO: rỗng; lấy gốc từ cuối; dựng PHẢI rồi TRÁI
        pass
    return build(0, len(inorder) - 1)


# assert serialize(exercise_2_build_tree([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])) == [3, 9, 20, None, None, 15, 7]
# assert serialize(exercise_2_build_tree([-1], [-1])) == [-1]

## Bài 3 — Biến thể: suy ra preorder từ inorder + postorder (không dựng cây)

```python
preorder_from([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) -> [3, 9, 20, 15, 7]
```

Gợi ý nhanh: như `build_hash`, nhưng thay vì tạo nút thì ghi giá trị gốc. Vì đọc theo thứ tự gốc → phải → trái, kết quả cần **sắp xếp lại** — hoặc dùng cách khoảng (mục 8) để duyệt gốc → trái → phải.

In [28]:
def exercise_3_preorder_from(inorder: List[int], postorder: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_3_preorder_from([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) == [3, 9, 20, 15, 7]
# assert exercise_3_preorder_from([1], [1]) == [1]

## Bài 4 — Biến thể: Construct from Preorder and Postorder (LeetCode #889)

Với cây nhị phân **đầy đủ** (mỗi nút có 0 hoặc 2 con), preorder + postorder xác định duy nhất cây. Dựng nó.

```python
from_pre_post([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1]) -> [1, 2, 3, 4, 5, 6, 7]
```

Gợi ý nhanh: gốc `pre[0]`; gốc cây trái là `pre[1]`; vị trí của `pre[1]` trong postorder cho biết kích thước cây trái.

In [29]:
def exercise_4_from_pre_post(preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# assert serialize(exercise_4_from_pre_post([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1])) == [1, 2, 3, 4, 5, 6, 7]
# assert serialize(exercise_4_from_pre_post([1], [1])) == [1]

## Bài 5 — Mini challenge: kiểm tra hai traversal có hợp lệ không

Cho `inorder` và `postorder` (giá trị phân biệt, cùng tập), kiểm tra có tồn tại cây nhị phân nào cho đúng hai traversal đó không.

```python
is_valid_pair([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) -> True
is_valid_pair([1, 2, 3], [3, 1, 2])                  -> False
```

Gợi ý nhanh: dùng cách khoảng (mục 8); ở mỗi bước kiểm tra gốc `postorder[post_hi]` có nằm **trong** đoạn inorder hiện tại không.

In [30]:
def exercise_5_is_valid_pair(inorder: List[int], postorder: List[int]) -> bool:
    # TODO
    pass


# assert exercise_5_is_valid_pair([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) is True
# assert exercise_5_is_valid_pair([1, 2, 3], [3, 1, 2]) is False
# assert exercise_5_is_valid_pair([1, 2], [1, 2]) is True

# 14. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_build` sau khi tự làm.

**Bài 3**
- Mức 1: dùng `build(in_lo, in_hi, post_lo, post_hi)`; ghi gốc **trước** khi đệ quy trái rồi phải.

**Bài 4**
- Mức 1: `build(pre_lo, pre_hi, post_lo)`: nếu đoạn rỗng thì `None`; nếu 1 phần tử thì lá.
- Mức 2: `L = post_pos[pre[pre_lo + 1]] - post_lo + 1` là kích thước cây trái.

**Bài 5**
- Mức 1: nếu `k = pos[gốc]` không thuộc `[in_lo, in_hi]` thì `False`.
- Mức 2: kiểm tra đệ quy cả hai phía.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [31]:
_ = trace_build([2, 1, 3], [2, 3, 1])
print()
_ = trace_build([1, 2, 3, 4], [1, 3, 4, 2])

postorder[2] = 1 là gốc của inorder[2, 1, 3] -> phải [3], trái [2]
  postorder[1] = 3 là gốc của inorder[3] -> phải [], trái []
  postorder[0] = 2 là gốc của inorder[2] -> phải [], trái []
Cây: [1, 2, 3]

postorder[3] = 2 là gốc của inorder[1, 2, 3, 4] -> phải [3, 4], trái [1]
  postorder[2] = 4 là gốc của inorder[3, 4] -> phải [], trái [3]
    postorder[1] = 3 là gốc của inorder[3] -> phải [], trái []
  postorder[0] = 1 là gốc của inorder[1] -> phải [], trái []
Cây: [2, 1, 4, None, None, 3]


## Đáp án Bài 2

In [32]:
def answer_2_build_tree(inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    return build_hash(inorder, postorder)

assert serialize(answer_2_build_tree([9, 3, 15, 20, 7], [9, 15, 7, 20, 3])) == [3, 9, 20, None, None, 15, 7]
assert serialize(answer_2_build_tree([-1], [-1])) == [-1]
print("Bài 2 PASS (giống build_hash ở mục 6)")

Bài 2 PASS (giống build_hash ở mục 6)


## Đáp án Bài 3

In [33]:
def preorder_of(root):
    return [root.val] + preorder_of(root.left) + preorder_of(root.right) if root else []

def answer_3_preorder_from(inorder: List[int], postorder: List[int]) -> List[int]:
    pos = {v: i for i, v in enumerate(inorder)}
    out = []
    def walk(in_lo, in_hi, post_lo, post_hi):
        if in_lo > in_hi:
            return
        root_val = postorder[post_hi]
        out.append(root_val)
        k = pos[root_val]
        left_size = k - in_lo
        walk(in_lo, k - 1, post_lo, post_lo + left_size - 1)
        walk(k + 1, in_hi, post_lo + left_size, post_hi - 1)
    walk(0, len(inorder) - 1, 0, len(postorder) - 1)
    return out

assert answer_3_preorder_from([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) == [3, 9, 20, 15, 7]
assert answer_3_preorder_from([1], [1]) == [1]
for _ in range(2000):
    t = random_tree(random.randint(1, 12))
    assert answer_3_preorder_from(inorder_of(t), postorder_of(t)) == preorder_of(t)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [34]:
def answer_4_from_pre_post(preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
    post_pos = {v: i for i, v in enumerate(postorder)}
    def build(pre_lo, pre_hi, post_lo):
        if pre_lo > pre_hi:
            return None
        root = TreeNode(preorder[pre_lo])
        if pre_lo == pre_hi:
            return root
        L = post_pos[preorder[pre_lo + 1]] - post_lo + 1
        root.left = build(pre_lo + 1, pre_lo + L, post_lo)
        root.right = build(pre_lo + L + 1, pre_hi, post_lo + L)
        return root
    return build(0, len(preorder) - 1, 0)

assert serialize(answer_4_from_pre_post([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1])) == [1, 2, 3, 4, 5, 6, 7]
assert serialize(answer_4_from_pre_post([1], [1])) == [1]

def random_full_tree(n_internal, vals):
    if n_internal == 0:
        return TreeNode(next(vals))
    k = random.randint(0, n_internal - 1)
    node = TreeNode(next(vals))
    node.left = random_full_tree(k, vals)
    node.right = random_full_tree(n_internal - 1 - k, vals)
    return node

for _ in range(2000):
    m = random.randint(0, 7)
    t = random_full_tree(m, iter(random.sample(range(1000), 2 * m + 1)))
    assert serialize(answer_4_from_pre_post(preorder_of(t), postorder_of(t))) == serialize(t)
print("Bài 4 PASS (2000 cây đầy đủ ngẫu nhiên)")

Bài 4 PASS (2000 cây đầy đủ ngẫu nhiên)


## Đáp án Bài 5

In [35]:
def answer_5_is_valid_pair(inorder: List[int], postorder: List[int]) -> bool:
    pos = {v: i for i, v in enumerate(inorder)}
    def ok(in_lo, in_hi, post_lo, post_hi):
        if in_lo > in_hi:
            return True
        k = pos[postorder[post_hi]]
        if not in_lo <= k <= in_hi:
            return False
        left_size = k - in_lo
        return ok(in_lo, k - 1, post_lo, post_lo + left_size - 1) and \
               ok(k + 1, in_hi, post_lo + left_size, post_hi - 1)
    return ok(0, len(inorder) - 1, 0, len(postorder) - 1)

assert answer_5_is_valid_pair([9, 3, 15, 20, 7], [9, 15, 7, 20, 3]) is True
assert answer_5_is_valid_pair([1, 2, 3], [3, 1, 2]) is False
assert answer_5_is_valid_pair([1, 2], [1, 2]) is True

import itertools
for n in range(1, 6):
    vals = list(range(n))
    valid_posts = set()
    def all_trees(lo, hi):
        if lo > hi:
            return [None]
        out = []
        for r in range(lo, hi + 1):
            for L in all_trees(lo, r - 1):
                for R in all_trees(r + 1, hi):
                    out.append(TreeNode(r, L, R))
        return out
    for tr in all_trees(0, n - 1):                 # mọi cây có inorder = 0..n-1
        valid_posts.add(tuple(postorder_of(tr)))
    for perm in itertools.permutations(vals):
        assert answer_5_is_valid_pair(vals, list(perm)) == (perm in valid_posts)
print("Bài 5 PASS (vét cạn mọi postorder với n <= 5)")

Bài 5 PASS (vét cạn mọi postorder với n <= 5)


# 16. Tổng kết

> Gốc = `postorder` từ **cuối** lên; vị trí của nó trong inorder (hash map) chia trái/phải. Con trỏ chạy ngược đọc "gốc → phải → trái", nên **dựng cây phải trước**. Hoặc truyền khoảng của cả hai mảng để khỏi lo thứ tự.

---

## Checklist trước khi nộp

- [ ] Gốc lấy ở **cuối** đoạn postorder?
- [ ] Dùng con trỏ ngược thì dựng **phải trước trái**?
- [ ] Cắt postorder: trái `[:k]`, phải `[k:-1]`?
- [ ] Dựng `pos` một lần?